In [0]:
%sql
create catalog if not exists usecasecatalog;
create schema if not exists usecasecatalog.usecaseschema;
create volume if not exists usecasecatalog.usecaseschema.usecasevolume; 

In [0]:
# Use Case-1
# The data team has provided two CSV files — customer_data.csv and sales_data.csv. Load them into Spark DataFrames, infer schemas automatically, and print the schema along with the first 5 rows of each.

cust_df = spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/customer_data*",inferSchema=True,header=True)
sales_df = spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/sales_data*",inferSchema=True,header=True)
cust_df.printSchema()
cust_df.show(5,False)
sales_df.printSchema()
sales_df.show(5,False)

In [0]:
# Use case - 2
# The marketing team wants to know the total revenue generated by the 'Clothing' category for each month. Revenue = quantity × price.

sales_df.filter(sales_df.category == "Clothing").withColumn("Revenue",sales_df.quantity * sales_df.price).show(10)


In [0]:
# Use case - 3
# Who is the best customer? Find the customer who has spent the most money overall. Include their customer_id and name (from the customer table). Show the top 5.

from pyspark.sql.functions import max,col 

sales_df.groupBy(col("customer_id"))\
        .agg(max(col("price"))\
        .alias("max_spent"))\
        .orderBy(col("max_spent").desc())\
        .show(5)



In [0]:
# Use case - 4
#Generate a customer revenue summary.
#For each customer, show their total revenue and classify them as 'High Value' (≥ 5000), 'Mid Value' (≥ 1000), or 'Low Value' (< 1000)

from pyspark.sql.functions import when,col 

sales_df1= sales_df.select("customer_id","category","quantity","price") 

sales_df2 = sales_df1.withColumn("Revenue",sales_df.quantity * sales_df.price)

sales_df2.select(
                "*", 
                when(col("Revenue")>=5000,"High Value")
               .when((col("Revenue")>=1000) & (col("Revenue")<=5000),"Mid Value") 
               .when(col("Revenue")<1000,"Low Value")
               .otherwise("Unknown").alias("Revenue Category")
               )\
            .show(50)


In [0]:
# Use case - 5
# Is there a significant spending difference between male and female customers? Compute average transaction value, total revenue, and transaction count by gender.

from pyspark.sql.functions import col,avg,count,sum

sales_df1= sales_df.select("customer_id","price","quantity").withColumn("Total Revenue",sales_df.quantity * sales_df.price)
cust_df1 = cust_df.select("customer_id","gender")
sales_df3 = sales_df1.join(cust_df1,on ="customer_id", how ="inner")
sales_df3.groupBy("gender")\
         .agg(avg(col("price")).alias("Avg Transaction Value")\
              ,count(col("price")).alias("Transaction count")\
              ,sum("Total Revenue").alias("Total Revenue"))\
             .show()

In [0]:
# Use case - 6
# The finance team wants to know which payment method is most popular for each product category. Show the count of transactions per payment method per category.

from pyspark.sql.functions import count,col


cust_df1 = cust_df.join(sales_df,on = "customer_id",how ="inner")
cust_df2= cust_df1.select("payment_method","invoice_no","category")

# using pivot
cust_df2.groupBy("category")\
        .pivot("payment_method")\
        .count()\
        .show()

# just the count 
cust_df2.groupBy("category","payment_method")\
        .agg(count(col("invoice_no")).alias("Invoice count"))\
        .show()


In [0]:
# Use case - 7
# Data integrity check: Are there any transactions with a customer_id that does not exist in the customer master table? Identify and count orphaned transactions.

#left_anti is a join type in Spark that returns only the rows from the left DataFrame that do not have a matching row in the right DataFrame.
#It's commonly used to find:
#Records with no matching key.
#Orphan records.
#Data quality issues (missing master data).

from pyspark.sql.functions import col
cust_df1 = cust_df.join(sales_df,on ="customer_id",how ="left_anti")
cust_df1.show()



In [0]:
# Use case -8
# The marketing team wants to target ads by age group. Bucket customers into: Teens (< 20), Young Adults (20–35), Adults (36–50), Seniors (50+). Which segment generates the most revenue?

from pyspark.sql.functions import when,col,sum 

cust_df1 = cust_df.select (
                        "*", 
                        when(col("age")<20 ,"Teens")
                       .when((col("age")>=20) & (col("age")<=35),"Young Adults")
                       .when((col("age")>=36) & (col("age")<=50),"Adults")
                       .when ((col("age")>50),"Seniors")
                       .otherwise("Unknown")
                       .alias("Age Group")
                       )

most_revenue_df = cust_df1.join(sales_df,on ="customer_id",how ="inner")

most_revenue_df.groupBy("Age Group")\
               .agg(sum("price").alias("Total Revenue"))\
               .orderBy(col("Total Revenue").desc())\
               .show()

In [0]:
# Use case -9
# The retention team wants to run a win-back campaign. Identify all customers whose most recent purchase is more than 90 days before the latest invoice date in the dataset. consider max_date from the table as current_date

cust_df1 = cust_df.join(sales_df,on ="customer_id",how ="inner")
cust_df1.createOrReplaceTempView("cust_df1")
most_recent_purchase = spark.sql("select customer_id,max(invoice_date) as max_date from cust_df1 group by customer_id")
most_recent_purchase.createOrReplaceTempView("most_recent_purchase")
max_dt_purchase = spark.sql("select max(invoice_date) as max_date from cust_df1")
max_dt_purchase.createOrReplaceTempView("max_dt_purchase")
win_back = spark.sql("select customer_id,max_date, datediff((select max(max_date) as max_date from most_recent_purchase),max_date) as date_diff from most_recent_purchase where max_date < (select max(max_date) as max_date from max_dt_purchase) and  datediff((select max(max_date) as max_date from most_recent_purchase),max_date)>90")
win_back.show()

In [0]:
# Use case - 10
# Create a clean, analysis-ready master dataset by joining customer and transaction tables. Derive revenue, month, day-of-week, and age group. Handle nulls. Persist as a Parquet table.

from pyspark.sql.functions import col,month,dayofweek,when

sales_df1 = sales_df\
    .withColumn("Revenue",sales_df.quantity * sales_df.price)\
    .withColumn("Month",month(col("invoice_date")))\
    .withColumn("Day_of_week",dayofweek(col("invoice_date")))
cust_df1 = cust_df.join(sales_df1,on ="customer_id",how ="inner")

final_df = cust_df1.select("customer_id","Revenue","Month","Day_of_week",
                    when(col("age")<20 ,"Teens")\
                   .when((col("age")>=20) & (col("age")<=35),"Young Adults")\
                   .when((col("age")>=36) & (col("age")<=50),"Adults")\
                   .when ((col("age")>50),"Seniors")\
                  .otherwise("Unknown")\
                  .alias("Age Group")
                )

final_df.write.mode("overwrite").parquet("/Volumes/usecasecatalog/usecaseschema/usecasevolume/sales_parquet")

In [0]:

# Use case - 11
# What is the gender distribution across different product categories
cust_df1 = cust_df.join(sales_df,on ="customer_id",how ="inner")
cust_df1.groupBy("gender").pivot("category").count().show()


In [0]:
# Use case - 12
# What is the total revenue generated in the year 2022

sales_df1 = sales_df.withColumn("Revenue",sales_df.quantity * sales_df.price).alias("Total_Revenue")
sales_df1.write.mode("overwrite").saveAsTable("usecasecatalog.usecaseschema.sales_revenue")
spark.sql("select * from usecasecatalog.usecaseschema.sales_revenue where year(invoice_date) = '2022'").show()
